<h1> Step 1 – Decision Trees </h1>

In [35]:
# import seaborn as sns
from matplotlib import pyplot as plt
from sklearn import datasets
from sklearn.tree import DecisionTreeClassifier 
from sklearn import tree
import pandas as pd
import dtreeviz

<h3> Data set link </h3>
<a href="https://www.kaggle.com/datasets/sumedh1507/asthma-dataset">Asthma Risk & Severity Dataset</a>


In [36]:
# load the data
df = pd.read_csv("synthetic_asthma_dataset.csv")


# let's quickly see the first 5 rows of data
df

,Patient_ID,Age,Gender,BMI,Smoking_Status,Family_History,Allergies,Air_Pollution_Level,Physical_Activity_Level,Occupation_Type,Comorbidities,Medication_Adherence,Number_of_ER_Visits,Peak_Expiratory_Flow,FeNO_Level,Has_Asthma,Asthma_Control_Level
0,ASTH100000,52,Female,27.6,Former,1,NaN,Moderate,Sedentary,Outdoor,Diabetes,0.38,0,421.0,46.0,0,NaN
1,ASTH100001,15,Male,24.6,Former,0,Dust,Low,Moderate,Indoor,Both,0.60,2,297.6,22.9,0,NaN
2,ASTH100002,72,Female,17.6,Never,0,NaN,Moderate,Moderate,Indoor,NaN,0.38,0,303.3,15.3,0,NaN
3,ASTH100003,61,Male,16.8,Never,0,Multiple,High,Sedentary,Outdoor,Both,0.60,1,438.0,40.1,1,Poorly Controlled
4,ASTH100004,21,Male,30.2,Never,0,NaN,Moderate,Active,Indoor,NaN,0.82,3,535.0,27.7,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,ASTH109995,70,Male,25.0,Never,0,NaN,Low,Sedentary,Indoor,NaN,0.67,0,580.6,18.7,0,NaN
9996,ASTH109996,78,Female,24.8,Never,0,Pollen,Low,Moderate,Indoor,Diabetes,0.72,1,417.6,40.8,0,NaN
9997,ASTH109997,58,Male,30.1,Former,1,Pollen,Low,Moderate,Indoor,NaN,0.28,0,459.1,20.3,1,Not Controlled
9998,ASTH109998,88,Female,31.2,Former,0,Pollen,Moderate,Moderate,Indoor,NaN,0.44,0,415.9,25.0,0,NaN


In [37]:
# Check value counts for Has_Asthma:
df['Has_Asthma'].value_counts()

Has_Asthma
0    7567
1    2433
Name: count, dtype: int64

In [38]:
# Check total no of records
df.shape

(10000, 17)

In [39]:
# drop Patient_ID field since it is not required
df.drop(columns=['Patient_ID'], inplace=True)
df.head()

,Age,Gender,BMI,Smoking_Status,Family_History,Allergies,Air_Pollution_Level,Physical_Activity_Level,Occupation_Type,Comorbidities,Medication_Adherence,Number_of_ER_Visits,Peak_Expiratory_Flow,FeNO_Level,Has_Asthma,Asthma_Control_Level
0,52,Female,27.6,Former,1,NaN,Moderate,Sedentary,Outdoor,Diabetes,0.38,0,421.0,46.0,0,NaN
1,15,Male,24.6,Former,0,Dust,Low,Moderate,Indoor,Both,0.60,2,297.6,22.9,0,NaN
2,72,Female,17.6,Never,0,NaN,Moderate,Moderate,Indoor,NaN,0.38,0,303.3,15.3,0,NaN
3,61,Male,16.8,Never,0,Multiple,High,Sedentary,Outdoor,Both,0.60,1,438.0,40.1,1,Poorly Controlled
4,21,Male,30.2,Never,0,NaN,Moderate,Active,Indoor,NaN,0.82,3,535.0,27.7,0,NaN


In [40]:
# check duplicates and missing values
df.duplicated().sum()

np.int64(0)

In [41]:
# do we have missing values? apparently not
# dataset can't have any missing values when passing the data
# to the machine learning algorithm
df.isna().sum()

Age                           0
Gender                        0
BMI                           0
Smoking_Status                0
Family_History                0
Allergies                  2936
Air_Pollution_Level           0
Physical_Activity_Level       0
Occupation_Type               0
Comorbidities              4967
Medication_Adherence          0
Number_of_ER_Visits           0
Peak_Expiratory_Flow          0
FeNO_Level                    0
Has_Asthma                    0
Asthma_Control_Level       7567
dtype: int64

In [42]:
# Drop the columns Comorbidities, Asthma_Control_Level
df = df.drop(columns=['Comorbidities', 'Asthma_Control_Level'])
df.isna().sum()

Age                           0
Gender                        0
BMI                           0
Smoking_Status                0
Family_History                0
Allergies                  2936
Air_Pollution_Level           0
Physical_Activity_Level       0
Occupation_Type               0
Medication_Adherence          0
Number_of_ER_Visits           0
Peak_Expiratory_Flow          0
FeNO_Level                    0
Has_Asthma                    0
dtype: int64

In [43]:
# Check value counts for Allergies:
df['Allergies'].value_counts()


Allergies
Dust        2479
Pollen      1999
Pets        1585
Multiple    1001
Name: count, dtype: int64

In [44]:
# Example: replace NaN with None in Allergies
df['Allergies'] = df['Allergies'].fillna("None")


In [45]:
# Check for missing values again
df.isna().sum()

Age                        0
Gender                     0
BMI                        0
Smoking_Status             0
Family_History             0
Allergies                  0
Air_Pollution_Level        0
Physical_Activity_Level    0
Occupation_Type            0
Medication_Adherence       0
Number_of_ER_Visits        0
Peak_Expiratory_Flow       0
FeNO_Level                 0
Has_Asthma                 0
dtype: int64

In [46]:
# Check value counts for Allergies:
df['Allergies'].value_counts()

Allergies
None        2936
Dust        2479
Pollen      1999
Pets        1585
Multiple    1001
Name: count, dtype: int64

In [47]:
# converts Allergies into binary since it is a nominal category

from sklearn.preprocessing import OneHotEncoder
variables = ['Allergies']
# use encoder
encoder = OneHotEncoder(sparse_output=False).set_output(transform="pandas")
one_hot_encoded = encoder.fit_transform(df[variables]).astype(int)
df = pd.concat([df,one_hot_encoded],axis=1).drop(columns=variables)
df.head()

,Age,Gender,BMI,Smoking_Status,Family_History,Air_Pollution_Level,Physical_Activity_Level,Occupation_Type,Medication_Adherence,Number_of_ER_Visits,Peak_Expiratory_Flow,FeNO_Level,Has_Asthma,Allergies_Dust,Allergies_Multiple,Allergies_None,Allergies_Pets,Allergies_Pollen
0,52,Female,27.6,Former,1,Moderate,Sedentary,Outdoor,0.38,0,421.0,46.0,0,0,0,1,0,0
1,15,Male,24.6,Former,0,Low,Moderate,Indoor,0.60,2,297.6,22.9,0,1,0,0,0,0
2,72,Female,17.6,Never,0,Moderate,Moderate,Indoor,0.38,0,303.3,15.3,0,0,0,1,0,0
3,61,Male,16.8,Never,0,High,Sedentary,Outdoor,0.60,1,438.0,40.1,1,0,1,0,0,0
4,21,Male,30.2,Never,0,Moderate,Active,Indoor,0.82,3,535.0,27.7,0,0,0,1,0,0


In [48]:
df.head()

,Age,Gender,BMI,Smoking_Status,Family_History,Air_Pollution_Level,Physical_Activity_Level,Occupation_Type,Medication_Adherence,Number_of_ER_Visits,Peak_Expiratory_Flow,FeNO_Level,Has_Asthma,Allergies_Dust,Allergies_Multiple,Allergies_None,Allergies_Pets,Allergies_Pollen
0,52,Female,27.6,Former,1,Moderate,Sedentary,Outdoor,0.38,0,421.0,46.0,0,0,0,1,0,0
1,15,Male,24.6,Former,0,Low,Moderate,Indoor,0.60,2,297.6,22.9,0,1,0,0,0,0
2,72,Female,17.6,Never,0,Moderate,Moderate,Indoor,0.38,0,303.3,15.3,0,0,0,1,0,0
3,61,Male,16.8,Never,0,High,Sedentary,Outdoor,0.60,1,438.0,40.1,1,0,1,0,0,0
4,21,Male,30.2,Never,0,Moderate,Active,Indoor,0.82,3,535.0,27.7,0,0,0,1,0,0


In [49]:
df.shape

(10000, 18)

In [50]:
# Check value counts for Gender:
df['Gender'].value_counts()

Gender
Female    4814
Male      4786
Other      400
Name: count, dtype: int64

In [51]:
# converts Gender into binary since it is a nominal category
variables 
from sklearn.preprocessing import OneHotEncoder
variables = ['Gender']
# use encoder
encoder = OneHotEncoder(sparse_output=False).set_output(transform="pandas")
one_hot_encoded = encoder.fit_transform(df[variables]).astype(int)
df = pd.concat([df,one_hot_encoded],axis=1).drop(columns=variables)
df.head()

,Age,BMI,Smoking_Status,Family_History,Air_Pollution_Level,Physical_Activity_Level,Occupation_Type,Medication_Adherence,Number_of_ER_Visits,Peak_Expiratory_Flow,FeNO_Level,Has_Asthma,Allergies_Dust,Allergies_Multiple,Allergies_None,Allergies_Pets,Allergies_Pollen,Gender_Female,Gender_Male,Gender_Other
0,52,27.6,Former,1,Moderate,Sedentary,Outdoor,0.38,0,421.0,46.0,0,0,0,1,0,0,1,0,0
1,15,24.6,Former,0,Low,Moderate,Indoor,0.60,2,297.6,22.9,0,1,0,0,0,0,0,1,0
2,72,17.6,Never,0,Moderate,Moderate,Indoor,0.38,0,303.3,15.3,0,0,0,1,0,0,1,0,0
3,61,16.8,Never,0,High,Sedentary,Outdoor,0.60,1,438.0,40.1,1,0,1,0,0,0,0,1,0
4,21,30.2,Never,0,Moderate,Active,Indoor,0.82,3,535.0,27.7,0,0,0,1,0,0,0,1,0


In [52]:
# Check value counts for Smoking_Status:
df['Smoking_Status'].value_counts()

Smoking_Status
Never      6070
Former     2487
Current    1443
Name: count, dtype: int64

In [53]:
# Map Smoking_Status - Ordinal categories
category_mapper = {
    'Current': 2,
    'Former': 1,
    'Never': 0
}

df['Smoking_Status'] = df['Smoking_Status'].replace(category_mapper)
df.head()

,Age,BMI,Smoking_Status,Family_History,Air_Pollution_Level,Physical_Activity_Level,Occupation_Type,Medication_Adherence,Number_of_ER_Visits,Peak_Expiratory_Flow,FeNO_Level,Has_Asthma,Allergies_Dust,Allergies_Multiple,Allergies_None,Allergies_Pets,Allergies_Pollen,Gender_Female,Gender_Male,Gender_Other
0,52,27.6,1,1,Moderate,Sedentary,Outdoor,0.38,0,421.0,46.0,0,0,0,1,0,0,1,0,0
1,15,24.6,1,0,Low,Moderate,Indoor,0.60,2,297.6,22.9,0,1,0,0,0,0,0,1,0
2,72,17.6,0,0,Moderate,Moderate,Indoor,0.38,0,303.3,15.3,0,0,0,1,0,0,1,0,0
3,61,16.8,0,0,High,Sedentary,Outdoor,0.60,1,438.0,40.1,1,0,1,0,0,0,0,1,0
4,21,30.2,0,0,Moderate,Active,Indoor,0.82,3,535.0,27.7,0,0,0,1,0,0,0,1,0


In [54]:
# Check value counts for Air_Pollution_Level:
df['Air_Pollution_Level'].value_counts()

Air_Pollution_Level
Moderate    4915
Low         2984
High        2101
Name: count, dtype: int64

In [55]:
# Map Air_Pollution_Level- Ordinal categories
category_mapper = {
    'High': 2,
    'Moderate': 1,
    'Low': 0
}

df['Air_Pollution_Level'] = df['Air_Pollution_Level'].replace(category_mapper)
df.head()

,Age,BMI,Smoking_Status,Family_History,Air_Pollution_Level,Physical_Activity_Level,Occupation_Type,Medication_Adherence,Number_of_ER_Visits,Peak_Expiratory_Flow,FeNO_Level,Has_Asthma,Allergies_Dust,Allergies_Multiple,Allergies_None,Allergies_Pets,Allergies_Pollen,Gender_Female,Gender_Male,Gender_Other
0,52,27.6,1,1,1,Sedentary,Outdoor,0.38,0,421.0,46.0,0,0,0,1,0,0,1,0,0
1,15,24.6,1,0,0,Moderate,Indoor,0.60,2,297.6,22.9,0,1,0,0,0,0,0,1,0
2,72,17.6,0,0,1,Moderate,Indoor,0.38,0,303.3,15.3,0,0,0,1,0,0,1,0,0
3,61,16.8,0,0,2,Sedentary,Outdoor,0.60,1,438.0,40.1,1,0,1,0,0,0,0,1,0
4,21,30.2,0,0,1,Active,Indoor,0.82,3,535.0,27.7,0,0,0,1,0,0,0,1,0


In [56]:
# Check value counts for Physical_Activity_Level:
df['Physical_Activity_Level'].value_counts()

Physical_Activity_Level
Sedentary    4062
Moderate     3909
Active       2029
Name: count, dtype: int64

In [57]:
# Map Physical_Activity_Level- Ordinal categories
category_mapper = {
    'Active': 2,
    'Moderate': 1,
    'Sedentary': 0
}

df['Physical_Activity_Level'] = df['Physical_Activity_Level'].replace(category_mapper)
df.head()

,Age,BMI,Smoking_Status,Family_History,Air_Pollution_Level,Physical_Activity_Level,Occupation_Type,Medication_Adherence,Number_of_ER_Visits,Peak_Expiratory_Flow,FeNO_Level,Has_Asthma,Allergies_Dust,Allergies_Multiple,Allergies_None,Allergies_Pets,Allergies_Pollen,Gender_Female,Gender_Male,Gender_Other
0,52,27.6,1,1,1,0,Outdoor,0.38,0,421.0,46.0,0,0,0,1,0,0,1,0,0
1,15,24.6,1,0,0,1,Indoor,0.60,2,297.6,22.9,0,1,0,0,0,0,0,1,0
2,72,17.6,0,0,1,1,Indoor,0.38,0,303.3,15.3,0,0,0,1,0,0,1,0,0
3,61,16.8,0,0,2,0,Outdoor,0.60,1,438.0,40.1,1,0,1,0,0,0,0,1,0
4,21,30.2,0,0,1,2,Indoor,0.82,3,535.0,27.7,0,0,0,1,0,0,0,1,0


In [58]:
# Check value counts for Occupation_Type:
df['Occupation_Type'].value_counts()

Occupation_Type
Indoor     7035
Outdoor    2965
Name: count, dtype: int64

In [59]:
from sklearn.preprocessing import LabelEncoder

# list all variables that can be binary-converted
variables = ['Occupation_Type']

# load the encoder
encoder = LabelEncoder()

# convert the listed variables
df[variables] = df[variables].apply(encoder.fit_transform)

df.head()

,Age,BMI,Smoking_Status,Family_History,Air_Pollution_Level,Physical_Activity_Level,Occupation_Type,Medication_Adherence,Number_of_ER_Visits,Peak_Expiratory_Flow,FeNO_Level,Has_Asthma,Allergies_Dust,Allergies_Multiple,Allergies_None,Allergies_Pets,Allergies_Pollen,Gender_Female,Gender_Male,Gender_Other
0,52,27.6,1,1,1,0,1,0.38,0,421.0,46.0,0,0,0,1,0,0,1,0,0
1,15,24.6,1,0,0,1,0,0.60,2,297.6,22.9,0,1,0,0,0,0,0,1,0
2,72,17.6,0,0,1,1,0,0.38,0,303.3,15.3,0,0,0,1,0,0,1,0,0
3,61,16.8,0,0,2,0,1,0.60,1,438.0,40.1,1,0,1,0,0,0,0,1,0
4,21,30.2,0,0,1,2,0,0.82,3,535.0,27.7,0,0,0,1,0,0,0,1,0


<h3>There's no ordinal or nominal categories here, we can proceed to X/y -phase</h3>

In [60]:
df.head()

,Age,BMI,Smoking_Status,Family_History,Air_Pollution_Level,Physical_Activity_Level,Occupation_Type,Medication_Adherence,Number_of_ER_Visits,Peak_Expiratory_Flow,FeNO_Level,Has_Asthma,Allergies_Dust,Allergies_Multiple,Allergies_None,Allergies_Pets,Allergies_Pollen,Gender_Female,Gender_Male,Gender_Other
0,52,27.6,1,1,1,0,1,0.38,0,421.0,46.0,0,0,0,1,0,0,1,0,0
1,15,24.6,1,0,0,1,0,0.60,2,297.6,22.9,0,1,0,0,0,0,0,1,0
2,72,17.6,0,0,1,1,0,0.38,0,303.3,15.3,0,0,0,1,0,0,1,0,0
3,61,16.8,0,0,2,0,1,0.60,1,438.0,40.1,1,0,1,0,0,0,0,1,0
4,21,30.2,0,0,1,2,0,0.82,3,535.0,27.7,0,0,0,1,0,0,0,1,0


<h3>X/y -split => Decision Tree</h3>

In [61]:
# X/y -split 
X = df.drop("Has_Asthma", axis=1)
y = df['Has_Asthma']

In [62]:
# create a decision tree classifier for the visualization
# and train the model with our data
clf = DecisionTreeClassifier()
model = clf.fit(X, y)

In [63]:
# visualize the decision tree
from sklearn.tree import export_graphviz
import subprocess
from sklearn import tree

# save the decision tree visualization into an svg-file

# NOTE! Always modify the class_names to match your data's TARGET VARIABLE OPTIONS
export_graphviz(clf, feature_names=X.columns, class_names=["No", "Yes"],
                filled=True, rounded=True, node_ids=True, out_file="synthetic_asthma_tree.dot")

# convert the DOT-file into SVG-file (which is supported by many tools
subprocess.call(["dot", "-Tsvg", "synthetic_asthma_tree.dot", '-o', 'synthetic_asthma_sk.svg'])

0

<h3>Version 2: Limit the amount of LEAVES in the tree (depth)</h3>

In [64]:
# visualize the decision tree
from sklearn.tree import export_graphviz
import subprocess
from sklearn import tree

# save the decision tree visualization into an svg-file
export_graphviz(clf, feature_names=X.columns, class_names=["No", "Yes"],
                filled=True, rounded=True, node_ids=True, out_file="synthetic_asthma_tree_limited.dot",
                max_depth=4)

# convert the DOT-file into SVG-file (which is supported by many tools
subprocess.call(["dot", "-Tsvg", "synthetic_asthma_tree_limited.dot", '-o', 'dt_synthetic_asthma_test_sk_limited.svg'])

0

In [65]:
# pip install dtreeviz
import dtreeviz

viz_model = dtreeviz.model(clf,
                           X_train=X, y_train=y,
                           feature_names=X.columns,
                           target_name="Decision",
                           class_names=["No", "Yes"])

# view in Jupyter notebook
# if decision tree is large, consider saving to .SVG -file
# and view with a web browser
viz_model.view(scale=1.5).save("dt_synthetic_asthma.svg")

c:\Users\Gayani\OneDrive - lucit\IntroToML\IntroductionToMLMethods\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names


In [66]:
# in order to limit the depth of the dtreeviz,
# you have to create a separate classifier, with max depth

# create a decision tree classifier for the visualization
# and train the model with our data
clf_limited = DecisionTreeClassifier(max_depth=4)
model_limited = clf_limited.fit(X, y)

# pip install dtreeviz
import dtreeviz

viz_model = dtreeviz.model(clf_limited,
                           X_train=X, y_train=y,
                           feature_names=X.columns,
                           target_name="Decision",
                           class_names=["No", "Yes"])

# view in Jupyter notebook
# if decision tree is large, consider saving to .SVG -file
# and view with a web browser
viz_model.view(scale=1.5).save("dt_synthetic_asthma_tree_limited.svg")

c:\Users\Gayani\OneDrive - lucit\IntroToML\IntroductionToMLMethods\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names


<h1 style="color: green;">Visualization</h1>

<!DOCTYPE html>
<html>
<head>
    <title>Asthma Prediction - Decision Tree</title>
</head>
<body>

<h2>Asthma Prediction Using Decision Tree</h2>

<p>
This predicts whether a person has asthma (Yes or No) based on several factors.
</p>

<h3>Main Factors</h3>
<ul>
    <li>Family History</li>
    <li>Physical Activity</li>
    <li>Air Pollution</li>
    <li>Allergies</li>
    <li>Smoking Status</li>
    <li>BMI (Body Mass Index)</li>
</ul>

<h3>Root Decision Node:</h3>
<p>
The first and most important condition:
</p>
<p><b>Family_History ≤ 0.5</b></p>

<ul>
    <li>If TRUE → Lower chance of asthma</li>
    <li>If FALSE → Higher chance of asthma</li>
</ul>


<h4>Family History:</h4>
<p>
People with a family history of asthma are more likely to have asthma.
</p>

<h4>Physical Activity:</h4>
<p>
Low physical activity increases the risk of asthma.
</p>

<h4>Air Pollution:</h4>
<p>
Higher air pollution levels increase asthma risk.
</p>

<h4>Allergies:</h4>
<p>
People with allergies are more likely to have asthma.
</p>

<h4>Smoking:</h4>
<p>
Smoking increases the likelihood of asthma.
</p>

<h4>BMI:</h4>
<p>
Higher BMI (overweight) is associated with higher asthma risk.
</p>

<h4>Example Rules:</h4>

<h4>High Risk</h4>
<p>
If a person has:
</p>
<ul>
    <li>Family history of asthma</li>
    <li>Low physical activity</li>
    <li>Allergies</li>
</ul>
<p><b>Prediction: Asthma = YES</b></p>

<h3>Low Risk</h3>
<p>
If a person has:
</p>
<ul>
    <li>No family history</li>
    <li>Low air pollution exposure</li>
    <li>Non-smoker</li>
    <li>Normal BMI</li>
</ul>
<p><b>Prediction: Asthma = NO</b></p>

<h2>Summary:</h2>
<p>
The model shows that asthma risk is influenced by genetic, environmental, 
and lifestyle factors. Family history, allergies, and physical activity 
are the most important predictors.
</p>

</body>
</html>
